<a href="https://colab.research.google.com/github/sowave06/AI-ML-DM-PROJECT/blob/main/CPROJECT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1: Install base packages (takes ~2-3 minutes)
!pip install -q transformers accelerate sentence-transformers torch huggingface_hub chromadb
print("✅ Core packages installed")

✅ Core packages installed


In [ ]:
# Cell 2: Login to Hugging Face
from huggingface_hub import login
login()  # You'll be prompted to enter your HF token

In [ ]:
# Cell 3: LLM Engine (Simulated)
class LLMEngine:
    def generate(self, prompt: str) -> str:
        """Simulates LLM response - replace with real API call later"""
        print(f"[LLM] Processing prompt ({len(prompt)} chars)...")
        return "The torchlight flickers. You see two paths ahead: A) Go left into darkness, B) Go right toward distant voices."

llm = LLMEngine()
print("✅ LLM Engine initialized")

✅ LLM Engine initialized


In [ ]:
# ✅ Cell 4: Memory Manager Setup
import chromadb
from sentence_transformers import SentenceTransformer
from collections import deque
from datetime import datetime, timezone
import uuid, time, os

# Simple mock LLM
class LLMEngine:
    def generate(self, prompt: str) -> str:
        print(f"🧠 Simulated LLM prompt: {prompt[:80]}...")
        return "Summary placeholder"

class MemoryManager:
    def __init__(self, chroma_dir="./chroma_db", collection_name="dm_memory"):
        self.short_term = deque(maxlen=5)
        self.embed_model = SentenceTransformer("all-MiniLM-L6-v2")
        self.client = None
        self.col = None
        self.chroma_dir = chroma_dir
        self.collection_name = collection_name
        self.llm = LLMEngine()
        self.turn_counter = 0

        self._initialize_chroma()

    def _initialize_chroma(self, retries=3, delay=1):
        """Initializes ChromaDB client with automatic fallback to in-memory mode."""
        for i in range(retries):
            try:
                # Ensure directory exists if using persistent mode
                if self.chroma_dir:
                    os.makedirs(self.chroma_dir, exist_ok=True)
                    self.client = chromadb.PersistentClient(path=self.chroma_dir)
                    print(f"📂 Using persistent ChromaDB client at {self.chroma_dir}")
                else:
                    self.client = chromadb.Client()
                    print("📂 Using in-memory ChromaDB client")

                # Try to load collection
                try:
                    self.col = self.client.get_collection(self.collection_name)
                    print(f"📁 Loaded existing collection: {self.collection_name}")
                except:
                    self.col = self.client.create_collection(self.collection_name)
                    print(f"📁 Created new collection: {self.collection_name}")
                break  # ✅ Success
            except Exception as e:
                msg = str(e)
                print(f"⚠ ChromaDB initialization failed ({msg})")
                if "unable to open database file" in msg or "readonly" in msg:
                    if i == retries - 1:
                        print("🚨 Persistent DB failed. Switching to in-memory mode.")
                        self.client = chromadb.Client()
                        self.col = self.client.create_collection(self.collection_name)
                        break
                    else:
                        print(f"⏳ Retrying in {delay}s...")
                        time.sleep(delay)
                else:
                    raise e

    def append_turn(self, role: str, text: str):
        self.short_term.append({
            "role": role,
            "text": text,
            "timestamp": datetime.now(timezone.utc).isoformat()
        })
        if role == "player":
            self.turn_counter += 1

    def get_short_term_context(self):
        return "\n".join(f"{t['role'].upper()}: {t['text']}" for t in self.short_term)

    def store_long_term(self, text: str, importance: float = 0.5):
        doc_id = str(uuid.uuid4())
        embedding = self.embed_model.encode([text]).tolist()
        metadata = {
            "stored_at": datetime.now(timezone.utc).isoformat(),
            "importance": importance,
            "type": "event"
        }
        self.col.add(ids=[doc_id], documents=[text],
                     embeddings=embedding, metadatas=[metadata])
        return doc_id

    def retrieve(self, query: str, top_k: int = 3):
        query_emb = self.embed_model.encode([query]).tolist()
        results = self.col.query(query_embeddings=query_emb, n_results=top_k)
        retrieved = []
        if results['ids'][0]:
            for i, doc_id in enumerate(results['ids'][0]):
                retrieved.append({
                    "id": doc_id,
                    "document": results['documents'][0][i],
                    "metadata": results['metadatas'][0][i]
                })
        return retrieved

# ✅ Initialize Memory Manager
mem = MemoryManager(chroma_dir="/tmp/chroma_db")  # use /tmp for guaranteed writable path
print("✅ Memory Manager initialized successfully")
print(f"📊 Current memory count: {mem.col.count()}")


📂 Using persistent ChromaDB client at /tmp/chroma_db
📁 Loaded existing collection: dm_memory
✅ Memory Manager initialized successfully
📊 Current memory count: 0


In [ ]:
(Fixed)
import chromadb
from sentence_transformers import SentenceTransformer
from collections import deque
from datetime import datetime, timezone
import uuid
import time

# Simulated LLM engine for completeness
class LLMEngine:
    def generate(self, prompt: str) -> str:
        print(f"🧠 Simulated LLM prompt: {prompt[:100]}...")
        return "Summary placeholder"

class MemoryManager:
    def __init__(self, chroma_dir="./chroma_db", collection_name="dm_memory"):
        self.short_term = deque(maxlen=5)
        self.embed_model = SentenceTransformer("all-MiniLM-L6-v2")
        self.client = None
        self.col = None
        self.chroma_dir = chroma_dir
        self.collection_name = collection_name

        # Initialize Chroma
        self._initialize_chroma()

        # LLM Engine + Counters
        self.llm = LLMEngine()
        self.turn_counter = 0

    def _initialize_chroma(self, retries=5, delay=1):
        """Initializes ChromaDB client with retries."""
        for i in range(retries):
            try:
                if self.chroma_dir is None:
                    self.client = chromadb.Client()  # In-memory
                    print("📂 Initialized in-memory ChromaDB client")
                else:
                    self.client = chromadb.PersistentClient(path=self.chroma_dir)
                    print(f"📂 Initialized persistent ChromaDB client at {self.chroma_dir}")

                # Try to load or create collection
                try:
                    self.col = self.client.get_collection(self.collection_name)
                    print(f"📁 Loaded existing collection: {self.collection_name}")
                except:
                    self.col = self.client.create_collection(self.collection_name)
                    print(f"📁 Created new collection: {self.collection_name}")
                break  # Success
            except Exception as e:
                if "readonly database" in str(e) and i < retries - 1:
                    print(f"⚠ ChromaDB read-only error, retrying in {delay}s...")
                    time.sleep(delay)
                else:
                    raise e

    def append_turn(self, role: str, text: str):
        self.short_term.append({
            "role": role,
            "text": text,
            "timestamp": datetime.now(timezone.utc).isoformat()
        })
        if role == "player":
            self.turn_counter += 1

    def get_short_term_context(self):
        return "\n".join(f"{t['role'].upper()}: {t['text']}" for t in self.short_term)

    def store_long_term(self, text: str, importance: float = 0.5, retries=5, delay=1):
        doc_id = str(uuid.uuid4())
        embedding = self.embed_model.encode([text]).tolist()
        metadata = {
            "stored_at": datetime.now(timezone.utc).isoformat(),
            "importance": importance,
            "type": "event"
        }
        for i in range(retries):
            try:
                self.col.add(ids=[doc_id], documents=[text],
                             embeddings=embedding, metadatas=[metadata])
                return doc_id
            except Exception as e:
                if "readonly database" in str(e) and i < retries - 1:
                    print(f"⚠ Read-only error, retrying in {delay}s...")
                    time.sleep(delay)
                else:
                    raise e
        return None

    def retrieve(self, query: str, top_k: int = 3):
        query_emb = self.embed_model.encode([query]).tolist()
        results = self.col.query(query_embeddings=query_emb, n_results=top_k)

        retrieved = []
        if results['ids'][0]:
            for i, doc_id in enumerate(results['ids'][0]):
                retrieved.append({
                    "id": doc_id,
                    "document": results['documents'][0][i],
                    "metadata": results['metadatas'][0][i] if results['metadatas'][0] else {}
                })
        return retrieved

    def compress_old_memories(self, keep_latest_n: int = 40):
        full = self.col.get()
        ids = full.get("ids", [])
        docs = full.get("documents", [])
        metas = full.get("metadatas", [])

        if len(ids) <= keep_latest_n:
            return

        # Sort by timestamp
        items = [(metas[i].get("stored_at", ""), ids[i], docs[i]) for i in range(len(ids))]
        items.sort()

        old_items = items[:10]
        old_texts = [it[2] for it in old_items]
        old_ids = [it[1] for it in old_items]

        summary = self.llm.generate("Summarize these events briefly:\n" + "\n".join(old_texts))

        self.col.delete(ids=old_ids)
        self.store_long_term(f"Summary: {summary}", importance=0.7)

        print(f"✅ Compressed {len(old_ids)} memories into 1 summary")

    def inspect_collection(self):
        try:
            return self.col.get(include=['documents', 'metadatas'])
        except Exception as e:
            print(f"Error inspecting collection: {e}")
            return None

# ✅ Initialize Memory Manager
mem = MemoryManager()
print("✅ Memory Manager initialized")
print(f"📊 Current long-term memory count: {mem.col.count()}")


InternalError: Database error: error returned from database: (code: 14) unable to open database file

In [ ]:
# Cell 4: Memory Manager Setup
import chromadb
from sentence_transformers import SentenceTransformer
from collections import deque
from datetime import datetime, timezone
import uuid
import time

class MemoryManager:
    def __init__(self, chroma_dir="./chroma_db", collection_name="dm_memory"):
        self.short_term = deque(maxlen=5)
        self.embed_model = SentenceTransformer("all-MiniLM-L6-v2")
        self.client = None  # Initialize client as None
        self.col = None  # Initialize collection as None
        self.chroma_dir = chroma_dir
        self.collection_name = collection_name

        self._initialize_chroma()

        # Initialize LLM Engine (using the simulated one for now)
        self.llm = LLMEngine()
        self.turn_counter = 0

    def _initialize_chroma(self, retries=5, delay=1):
        """Initializes ChromaDB client with retries."""
        for i in range(retries):
            try:
                if self.chroma_dir is None:
                    self.client = chromadb.Client() # In-memory client
                    print("📂 Initialized in-memory ChromaDB client")
                else:
                    self.client = chromadb.PersistentClient(path=self.chroma_dir)
                    print(f"📂 Initialized persistent ChromaDB client at {self.chroma_dir}")

                try:
                    self.col = self.client.get_collection(self.collection_name)
                    print(f"📂 Loaded existing collection: {self.collection_name}")
                except:
                    self.col = self.client.create_collection(self.collection_name)
                    print(f"📂 Created new collection: {self.collection_name}") # Fix: corrected f-string syntax
                break # Success, exit loop
            except Exception as e:
                if "readonly database" in str(e) and i < retries - 1:
                    print(f"⚠️ ChromaDB read-only error encountered during initialization. Retrying in {delay} seconds...")
                    time.sleep(delay)
                else:
                    raise e # Re-raise if not a read-only error or last retry

    def append_turn(self, role: str, text: str):
        self.short_term.append({"role": role, "text": text, "timestamp": datetime.now(timezone.utc).isoformat()})
        if role == "player":
            self.turn_counter += 1

    def get_short_term_context(self) -> str:
        return "\n".join(f"{t['role'].upper()}: {t['text']}" for t in self.short_term)

    def store_long_term(self, text: str, importance: float = 0.5, retries=5, delay=1):
        doc_id = str(uuid.uuid4())
        embedding = self.embed_model.encode([text]).tolist()
        metadata = {
            "stored_at": datetime.now(timezone.utc).isoformat(),
            "importance": importance,
            "type": "event"
        }
        for i in range(retries):
            try:
                self.col.add(ids=[doc_id], documents=[text], embeddings=embedding, metadatas=[metadata])
                return doc_id # Success, exit loop
            except Exception as e:
                if "readonly database" in str(e) and i < retries - 1:
                    print(f"⚠️ ChromaDB read-only error encountered during storage. Retrying in {delay} seconds...")
                    time.sleep(delay)
                else:
                    raise e # Re-raise if not a read-only error or last retry
        return None # Return None if all retries fail


    def retrieve(self, query: str, top_k: int = 3):
        query_emb = self.embed_model.encode([query]).tolist()
        results = self.col.query(query_embeddings=query_emb, n_results=top_k)

        retrieved = []
        if results['ids'][0]:
            for i, doc_id in enumerate(results['ids'][0]):
                retrieved.append({
                    "id": doc_id,
                    "document": results['documents'][0][i],
                    "metadata": results['metadatas'][0][i] if results['metadatas'][0] else {}
                })
        return retrieved

    def compress_old_memories(self, keep_latest_n: int = 40):
        """Summarize old memories to save space"""
        full = self.col.get()
        ids = full.get("ids", [])
        docs = full.get("documents", [])
        metas = full.get("metadatas", [])

        if len(ids) <= keep_latest_n:
            return  # Nothing to compress

        # Sort by timestamp
        items = [(metas[i].get("stored_at", ""), ids[i], docs[i]) for i in range(len(ids))]
        items.sort()

        # Compress oldest 10 items
        old_items = items[:10]
        old_texts = [item[2] for item in old_items]
        old_ids = [item[1] for item in old_ids]

        # Create summary
        # Ensure llm is initialized before calling generate
        if not hasattr(self, 'llm'):
             self.llm = LLMEngine() # Or RealLLMEngine() depending on your setup
             print("Warning: LLM not initialized in compress_old_memories, using default.")

        summary = self.llm.generate(f"Summarize these events briefly:\n" + "\n".join(old_texts))

        # Delete old, store summary
        self.col.delete(ids=old_ids)
        self.store_long_term(f"Summary: {summary}", importance=0.7)

        print(f"✅ Compressed {len(old_ids)} memories into 1 summary")

    def inspect_collection(self):
        """Retrieves and returns all items in the ChromaDB collection."""
        try:
            full = self.col.get(include=['documents', 'metadatas'])
            return full
        except Exception as e:
            print(f"Error inspecting collection: {e}")
            return None

# Initialize Memory Manager
mem = MemoryManager()
print("✅ Memory Manager initialized")
print(f"📊 Current long-term memory count: {mem.col.count()}")

InternalError: Database error: error returned from database: (code: 1) no such table: tenants

In [ ]:
# Initialize Memory Manager with in-memory ChromaDB
mem = MemoryManager(chroma_dir=None)
print("✅ Memory Manager initialized with in-memory database")
print(f"📊 Current long-term memory count: {mem.col.count()}")

📂 Initialized in-memory ChromaDB client
📂 Loaded existing collection: dm_memory
✅ Memory Manager initialized with in-memory database
📊 Current long-term memory count: 16


In [ ]:
# Cell 5: Handle Player Turns
def handle_player_turn(player_text: str):
    # 1. Add to short-term memory
    mem.append_turn("player", player_text)

    # 2. Retrieve relevant long-term memories
    # Using a higher top_k for initial retrieval before reranking
    retrieved = mem.retrieve(player_text, top_k=8)

    # 3. Build context
    short_ctx = mem.get_short_term_context()
    context_parts = ["--- Recent Turns ---", short_ctx]

    if retrieved:
        context_parts.append("\n--- Retrieved Memories ---")
        for r in retrieved:
            context_parts.append(f"• {r['document']}")

    context_parts.append(f"\n--- Player Action ---\n{player_text}\n")
    prompt = "\n".join(context_parts)

    # 4. Generate DM response
    dm_reply = mem.llm.generate(prompt)

    # 5. Store DM response
    mem.append_turn("dm", dm_reply)
    mem.store_long_term(f"Player: {player_text}\nDM: {dm_reply}", importance=0.6)

    # 6. Compress old memories if needed
    if mem.turn_counter % 10 == 0:
        mem.compress_old_memories()


    return {
        "dm_reply": dm_reply,
        "retrieved": retrieved,
        "prompt": prompt
    }

print("✅ RAG Pipeline ready")

✅ RAG Pipeline ready


In [ ]:
# Cell 6: Simple Test
print("🧪 Testing the DM system...\n")

# Test 1
response1 = handle_player_turn("I enter a dark tavern.")
print(f"DM: {response1['dm_reply']}\n")

# Test 2
response2 = handle_player_turn("I talk to the bartender.")
print(f"DM: {response2['dm_reply']}\n")

# Check memory
print(f"📊 Short-term memory:\n{mem.get_short_term_context()}\n")
print(f"💾 Long-term memory count: {mem.col.count()}")

🧪 Testing the DM system...

[LLM] Processing prompt (98 chars)...
DM: The torchlight flickers. You see two paths ahead: A) Go left into darkness, B) Go right toward distant voices.

[LLM] Processing prompt (424 chars)...
DM: The torchlight flickers. You see two paths ahead: A) Go left into darkness, B) Go right toward distant voices.

📊 Short-term memory:
PLAYER: I enter a dark tavern.
DM: The torchlight flickers. You see two paths ahead: A) Go left into darkness, B) Go right toward distant voices.
PLAYER: I talk to the bartender.
DM: The torchlight flickers. You see two paths ahead: A) Go left into darkness, B) Go right toward distant voices.

💾 Long-term memory count: 2


In [ ]:
# Cell 7: Simple interactive loop
print("🧙 AI Dungeon Master Started!")
print("Type 'quit' to exit\n")

while True:
    user_input = input("🗣️ You: ")

    if user_input.lower() in ['quit', 'exit', 'q']:
        print("\n👋 Adventure ended. Memories saved!")
        break

    if not user_input.strip():
        continue

    result = handle_player_turn(user_input)
    print(f"\n🤖 DM: {result['dm_reply']}\n")

🧙 AI Dungeon Master Started!
Type 'quit' to exit

🗣️ You: quit

👋 Adventure ended. Memories saved!


In [ ]:
# Cell 8: Install and setup Gradio
!pip install -q gradio

import gradio as gr

def chat_interface(user_input, history):
    if not user_input.strip():
        return history

    result = handle_player_turn(user_input)
    history.append((user_input, result['dm_reply']))
    return history

with gr.Blocks(title="AI Dungeon Master") as demo:
    gr.Markdown("# 🧙 AI Dungeon Master")

    chatbot = gr.Chatbot(label="Adventure Log", height=400)
    msg = gr.Textbox(label="Your Action", placeholder="What do you do?")
    send = gr.Button("Send")

    send.click(chat_interface, inputs=[msg, chatbot], outputs=[chatbot])
    msg.submit(chat_interface, inputs=[msg, chatbot], outputs=[chatbot])

demo.launch(share=True)

/tmp/ipython-input-2356002041.py:17: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(label="Adventure Log", height=400)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d8d0baed29771ecf53.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# Run this to clear GPU memory
import torch
torch.cuda.empty_cache()
import gc
gc.collect()

18491

In [ ]:
!nvidia-smi  # Check GPU usage
import psutil
print(f"RAM usage: {psutil.virtual_memory().percent}%")

/bin/bash: line 1: nvidia-smi: command not found
RAM usage: 34.6%


In [ ]:
import shutil
import os
import time # Import time module

# Remove the chroma_db directory if it exists
if os.path.exists('./chroma_db'):
    shutil.rmtree('./chroma_db')
    print("✅ Removed existing chroma_db directory")
    time.sleep(1) # Add a small delay

# Re-initialize Memory Manager
mem = MemoryManager()
print("✅ Memory Manager re-initialized")

✅ Removed existing chroma_db directory
⚠ ChromaDB initialization failed (Database error: error returned from database: (code: 1) no such table: tenants)


InternalError: Database error: error returned from database: (code: 1) no such table: tenants

In [ ]:
# Qwen model loading
qwen_model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-0.6B")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [ ]:
# Cell: Load Real Qwen Model
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

class RealLLMEngine:
    def __init__(self, model_name="Qwen/Qwen2-1.5B-Instruct"):  # Using smaller model for Colab
        print("🔄 Loading model (this takes 2-3 minutes)...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map="auto",
            torch_dtype=torch.float16  # Use half precision for Colab
        )
        print("✅ Model loaded!")

    def generate(self, prompt: str, max_tokens=150) -> str:
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        outputs = self.model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.8,
            do_sample=True
        )
        return self.tokenizer.decode(outputs[0], skip_special_tokens=True)

# Replace in MemoryManager.__init__:
# self.llm = RealLLMEngine()

In [ ]:
# Cell: Add Reranking
from sentence_transformers import CrossEncoder

class CrossReranker:
    def __init__(self):
        self.model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

    def rerank(self, query: str, candidates: list, top_n: int = 3):
        if not candidates:
            return []

        pairs = [(query, c['document']) for c in candidates]
        scores = self.model.predict(pairs)

        for c, score in zip(candidates, scores):
            c['rerank_score'] = float(score)

        return sorted(candidates, key=lambda x: x['rerank_score'], reverse=True)[:top_n]

# Update retrieve in MemoryManager:
def retrieve(self, query: str, top_k: int = 8):
    # ... existing code ...
    reranker = CrossReranker()
    return reranker.rerank(query, retrieved, top_n=3)

In [ ]:
# Cell: Memory Compression
def compress_old_memories(self, keep_latest_n: int = 40):
    """Summarize old memories to save space"""
    full = self.col.get()
    ids = full.get("ids", [])
    docs = full.get("documents", [])
    metas = full.get("metadatas", [])

    if len(ids) <= keep_latest_n:
        return  # Nothing to compress

    # Sort by timestamp
    items = [(metas[i].get("stored_at", ""), ids[i], docs[i]) for i in range(len(ids))]
    items.sort()

    # Compress oldest 10 items
    old_items = items[:10]
    old_texts = [item[2] for item in old_items]
    old_ids = [item[1] for item in old_items]

    # Create summary
    summary = self.llm.generate(f"Summarize these events briefly:\n" + "\n".join(old_texts))

    # Delete old, store summary
    self.col.delete(ids=old_ids)
    self.store_long_term(f"Summary: {summary}", importance=0.7)

    print(f"✅ Compressed {len(old_ids)} memories into 1 summary")

# Call this in handle_player_turn every 10 turns:
if mem.turn_counter % 10 == 0:
    mem.compress_old_memories()

In [ ]:
mem = MemoryManager(chroma_dir=None)  # Non-persistent, purely in-memory

📂 Initialized in-memory ChromaDB client
📂 Created new collection: dm_memory


In [ ]:
import shutil
import os

db_path = "chroma_db/tdnx6Am5ixnr"  # Replace with your actual path

# Remove old db
if os.path.exists(db_path):
    shutil.rmtree(db_path)

# Reinitialize mem (your MemoryManager)
mem = MemoryManager(chroma_dir=db_path)

📂 Created new collection: dm_memory


In [ ]:
# Cell: Evaluation Metrics
def test_system():
    """Run automated tests"""
    print("🧪 Running System Tests...\n")

    # Test 1: Memory Storage
    mem.store_long_term("Player found a golden key", importance=0.8)
    assert mem.col.count() > 0, "❌ Memory storage failed"
    print("✅ Test 1: Memory storage works")

    # Test 2: Retrieval
    results = mem.retrieve("golden key", top_k=3)
    assert len(results) > 0, "❌ Retrieval failed"
    assert "key" in results[0]['document'].lower(), "❌ Retrieval not relevant"
    print("✅ Test 2: Retrieval works")

    # Test 3: Short-term buffer
    for i in range(8):
        mem.append_turn("player", f"Action {i}")
    assert len(mem.short_term) == 5, "❌ Short-term buffer not rolling"
    print("✅ Test 3: Short-term buffer rolls correctly")

    # Test 4: RAG Pipeline
    response = handle_player_turn("I look around")
    assert 'dm_reply' in response, "❌ RAG pipeline failed"
    print("✅ Test 4: RAG pipeline works")

    print("\n✅ All tests passed!")

# Run tests
test_system()

🧪 Running System Tests...

✅ Test 1: Memory storage works
✅ Test 2: Retrieval works
✅ Test 3: Short-term buffer rolls correctly
[LLM] Processing prompt (204 chars)...
✅ Test 4: RAG pipeline works

✅ All tests passed!


In [ ]:
# Cell: Create Streamlit App file
%%writefile app.py
import streamlit as st
from your_notebook import mem, handle_player_turn  # Adjust import

st.title("🧙 AI Dungeon Master")

if "history" not in st.session_state:
    st.session_state.history = []

user_input = st.text_input("Your Action:")

if st.button("Send") and user_input:
    result = handle_player_turn(user_input)
    st.session_state.history.append((user_input, result['dm_reply']))

for user_msg, dm_msg in st.session_state.history:
    st.markdown(f"**You:** {user_msg}")
    st.markdown(f"**DM:** {dm_msg}")

# Run with: !streamlit run app.py & npx localtunnel --port 8501

Overwriting app.py


In [ ]:
%%writefile app.py
import streamlit as st
from your_notebook import mem, handle_player_turn  # Adjust import

st.title("🧙 AI Dungeon Master")

if "history" not in st.session_state:
    st.session_state.history = []

user_input = st.text_input("Your Action:")

if st.button("Send") and user_input:
    result = handle_player_turn(user_input)
    st.session_state.history.append((user_input, result['dm_reply']))

for user_msg, dm_msg in st.session_state.history:
    st.markdown(f"**You:** {user_msg}")
    st.markdown(f"**DM:** {dm_msg}")


Overwriting app.py


In [ ]:
# Cell: Add Metrics
import time

class Metrics:
    def __init__(self):
        self.total_turns = 0
        self.total_latency = 0
        self.retrieval_calls = 0

    def log_turn(self, latency: float):
        self.total_turns += 1
        self.total_latency += latency

    def log_retrieval(self):
        self.retrieval_calls += 1

    def report(self):
        avg_latency = self.total_latency / max(1, self.total_turns)
        print(f"📊 Metrics Report:")
        print(f"  - Total turns: {self.total_turns}")
        print(f"  - Avg latency: {avg_latency:.2f}s")
        print(f"  - Retrieval calls: {self.retrieval_calls}")

metrics = Metrics()

# Modify handle_player_turn to track time:
start = time.time()
# ... existing code ...
metrics.log_turn(time.time() - start)

In [ ]:
# Cell: Export/Import Memory
import json

def export_memory(filename="memory_backup.json"):
    """Export all memories to JSON"""
    full = mem.col.get()
    data = {
        "ids": full['ids'],
        "documents": full['documents'],
        "metadatas": full['metadatas']
    }
    with open(filename, 'w') as f:
        json.dump(data, f, indent=2)
    print(f"✅ Exported {len(full['ids'])} memories to {filename}")

def import_memory(filename="memory_backup.json"):
    """Import memories from JSON"""
    with open(filename, 'r') as f:
        data = json.load(f)

    embeddings = mem.embed_model.encode(data['documents']).tolist()
    mem.col.add(
        ids=data['ids'],
        documents=data['documents'],
        embeddings=embeddings,
        metadatas=data['metadatas']
    )
    print(f"✅ Imported {len(data['ids'])} memories")

# Usage:
# export_memory("my_game_session.json")

In [ ]:
# Cell: Install Streamlit
!pip install -q streamlit pyngrok

# Cell: Create Streamlit App
%%writefile app.py
import streamlit as st
from your_notebook import mem, handle_player_turn  # Adjust import

st.title("🧙 AI Dungeon Master")

if "history" not in st.session_state:
    st.session_state.history = []

user_input = st.text_input("Your Action:")

if st.button("Send") and user_input:
    result = handle_player_turn(user_input)
    st.session_state.history.append((user_input, result['dm_reply']))

for user_msg, dm_msg in st.session_state.history:
    st.markdown(f"**You:** {user_msg}")
    st.markdown(f"**DM:** {dm_msg}")

# Run with: !streamlit run app.py & npx localtunnel --port 8501

UsageError: Line magic function `%%writefile` not found.


In [ ]:
!pip install -q streamlit pyngrok


In [ ]:
!nohup streamlit run app.py &
!npx localtunnel --port 8501


nohup: appending output to 'nohup.out'
⠙⠹⠸⠼⠴⠦⠧⠇⠏your url is: https://curvy-teams-admire.loca.lt
^C


In [ ]:
# Cell: Install Gradio
!pip install -q gradio

# Cell: Create Gradio App
import gradio as gr
# from your_notebook import mem, handle_player_turn  # Adjust import - Removed this line

# Function to handle a single turn
def dm_turn(user_input, history):
    """
    user_input: string from player
    history: list of tuples [(user_msg, dm_msg), ...]
    """
    if user_input:
        # Directly use mem and handle_player_turn from the notebook environment
        result = handle_player_turn(user_input)
        history.append((user_input, result['dm_reply']))
    return history, history  # return for both output display and state update

# Define Gradio interface
with gr.Blocks() as demo:
    gr.Markdown("## 🧙 AI Dungeon Master")

    state = gr.State([])  # store conversation history

    chatbox = gr.Chatbot([], elem_id="chatbot")  # for chat display
    user_input = gr.Textbox(label="Your Action:")
    send_btn = gr.Button("Send")

    send_btn.click(
        dm_turn,
        inputs=[user_input, state],
        outputs=[chatbox, state]
    )

demo.launch()

/tmp/ipython-input-2564168782.py:26: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbox = gr.Chatbot([], elem_id="chatbot")  # for chat display


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://59b2bfdfa538aa4ab6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import json

demo_script = [
    {"user": "Hello, who are you?"},
    {"user": "Tell me a story about a dragon."},
    {"user": "Where was that dragon born?"},
    {"user": "Can you remember what color it was?"},
    {"user": "Now make the dragon meet a knight."},
    {"user": "What does the knight say to the dragon?"},
    {"user": "Remind me what happened earlier in the story."},
    {"user": "Change the ending to something funny."},
    {"user": "Summarize the whole story in one sentence."},
    {"user": "Goodbye!"}
]

with open("demo_script.json", "w") as f:
    json.dump(demo_script, f, indent=2)

print("✅ demo_script.json created!")

✅ demo_script.json created!


In [ ]:
import json
# from your_notebook import handle_player_turn # Removed this line

# Load demo script
with open("demo_script.json") as f:
    script = json.load(f)

session_id = "demo"

print("🚀 Starting 10-turn test...\n")
for i, turn in enumerate(script, 1):
    print(f"🧍 Turn {i} — User: {turn['user']}")
    # Directly use handle_player_turn from the notebook environment
    response = handle_player_turn(turn["user"]) # Pass user_input directly
    print(f"🤖 AI: {response}\n")

🚀 Starting 10-turn test...

🧍 Turn 1 — User: Hello, who are you?
[LLM] Processing prompt (458 chars)...
🤖 AI: {'dm_reply': 'The torchlight flickers. You see two paths ahead: A) Go left into darkness, B) Go right toward distant voices.', 'retrieved': [{'id': '5ca835d2-8b83-4187-867e-e45e0cbdbf51', 'document': 'Player: I look around\nDM: The torchlight flickers. You see two paths ahead: A) Go left into darkness, B) Go right toward distant voices.', 'metadata': {'stored_at': '2025-10-10T10:46:27.611845+00:00', 'importance': 0.6, 'type': 'event'}}, {'id': '891985de-f22c-4206-afca-25805724bc78', 'document': 'Player found a golden key', 'metadata': {'stored_at': '2025-10-10T10:46:27.539479+00:00', 'type': 'event', 'importance': 0.8}}], 'prompt': '--- Recent Turns ---\nPLAYER: Action 6\nPLAYER: Action 7\nPLAYER: I look around\nDM: The torchlight flickers. You see two paths ahead: A) Go left into darkness, B) Go right toward distant voices.\nPLAYER: Hello, who are you?\n\n--- Retrieved Memorie

In [ ]:
import pandas as pd

results = pd.DataFrame([
    {"tester_id": 1, "coherence": 5, "memory_recall": 4, "creativity": 5},
    {"tester_id": 2, "coherence": 4, "memory_recall": 5, "creativity": 4}
])
results.to_csv("tester_feedback.csv", index=False)
print("✅ tester_feedback.csv created!")

✅ tester_feedback.csv created!


In [ ]:
demo.launch(share=True)

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://59b2bfdfa538aa4ab6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
def handle_player_turn(user_input, session_id):
    # actual processing with memory
    # retrieves context from mem
    # calls LLM to generate a response
    response = llm.generate(user_input, context=mem.retrieve_memory(session_id))
    mem.store(session_id, user_input, response)
    return response

In [ ]:
display(mem.inspect_collection())  # should show empty dict at start

{'ids': ['891985de-f22c-4206-afca-25805724bc78',
  '5ca835d2-8b83-4187-867e-e45e0cbdbf51',
  '02593108-76a1-496f-ac45-7d4da6e207f8',
  'fb87b2be-3f51-4fbb-bb5a-4f5f3b8a0643',
  '72f4c916-deb7-44f4-9610-b433b69f13b1',
  '79855015-1447-464f-ac34-68b44a8cbec5',
  'b4a6c0cb-a0b1-45f5-a03f-3253f0c5b587',
  'f3dd28e4-3cad-4d89-b757-4cb4827cb6e3',
  '885b35b1-4df1-41c3-bf4d-277194a4ffe1',
  '5e49ca44-c976-41ab-a338-5f9509d561fb',
  '0c3fb94a-c2f4-4a7f-b92f-6b45b9a667b7',
  '50301b9b-5ed6-4b86-a110-f9cdbbb3dbc1',
  'd5c903b8-3274-4e13-b2a2-2145c218dae6',
  '19ceb25b-223d-4593-9355-8102c00898ff',
  'b39f2841-7f64-4e7e-92b8-f5f85b048f2b',
  '87a847c8-318b-494b-a10d-cf8eccdad889'],
 'embeddings': None,
 'documents': ['Player found a golden key',
  'Player: I look around\nDM: The torchlight flickers. You see two paths ahead: A) Go left into darkness, B) Go right toward distant voices.',
  'Player: Hello, who are you?\nDM: The torchlight flickers. You see two paths ahead: A) Go left into darkness, 

In [ ]:
# from memory_manager import MemoryManager # Removed import
mem = MemoryManager()

⚠ ChromaDB initialization failed (Database error: error returned from database: (code: 1) no such table: tenants)


InternalError: Database error: error returned from database: (code: 1) no such table: tenants

In [ ]:
import shutil, os

# Clean corrupted DB folder (if exists)
db_path = "./chroma_db"
if os.path.exists(db_path):
    shutil.rmtree(db_path)

print("🧹 Old ChromaDB directory removed")


🧹 Old ChromaDB directory removed


In [ ]:
mem = MemoryManager(chroma_dir="/tmp/chroma_db")


📂 Using persistent ChromaDB client at /tmp/chroma_db
📁 Loaded existing collection: dm_memory


In [ ]:
def _initialize_chroma(self, retries=3, delay=1):
    """Initializes ChromaDB client with robust recovery from corruption."""
    import shutil, os
    for i in range(retries):
        try:
            if self.chroma_dir:
                os.makedirs(self.chroma_dir, exist_ok=True)
                self.client = chromadb.PersistentClient(path=self.chroma_dir)
                print(f"📂 Using persistent ChromaDB client at {self.chroma_dir}")
            else:
                self.client = chromadb.Client()
                print("📂 Using in-memory ChromaDB client")

            try:
                self.col = self.client.get_collection(self.collection_name)
                print(f"📁 Loaded existing collection: {self.collection_name}")
            except:
                self.col = self.client.create_collection(self.collection_name)
                print(f"📁 Created new collection: {self.collection_name}")
            break

        except Exception as e:
            msg = str(e)
            print(f"⚠️ ChromaDB init error: {msg}")
            if ("no such table: tenants" in msg or
                "unable to open database file" in msg):
                print("💥 Detected corrupt or invalid DB. Rebuilding...")
                try:
                    shutil.rmtree(self.chroma_dir)
                except Exception:
                    pass
                time.sleep(delay)
                continue
            elif i == retries - 1:
                print("🚨 Persistent DB failed. Falling back to in-memory mode.")
                self.client = chromadb.Client()
                self.col = self.client.create_collection(self.collection_name)
                break
            else:
                time.sleep(delay)
